In [1]:
## Required libraries
import pandas as pd
import numpy as np
from fuzzywuzzy import fuzz
from fuzzywuzzy import process
import time
import os

pd.set_option("display.max_rows", 150)
pd.set_option("display.max_columns", 150)

In [2]:
# Enter the base data location
os.chdir('C:\\Users\\svi02\\Documents\\Python Scripts\\Matching\\new files')
base_xls = 'Copy of Accor Dep JAN.xlsx'
#base_xls = 'base_file.xlsx'
raw_url = '11-HRS_periodend_14_03_2020.csv'
#raw_url = "raw_file.csv"
result_url = "result_file_upd.csv"

In [3]:
#read data
base_org_df = pd.read_excel(base_xls, parse_dates=['Reservation Date from','Reservation Date to','timestamp Source','Inserted at','Modified at','Reservation Date','Reservation Time','Arrival Date','Departure Date'])
raw_org_df = pd.read_csv(raw_url, encoding = "ISO-8859-1", parse_dates=['Arrival','Departure','Pay Date'])

raw_org_col = list(raw_org_df.columns)

base_df=base_org_df.copy()
raw_df=raw_org_df.copy()

base_df.drop_duplicates(subset ="Reservation No_", inplace = True)     
raw_df.loc[raw_df["First Name"].notnull()==True, "Last Name"] = raw_df["Last Name"].str.cat(raw_df["First Name"], sep =" ", na_rep = " ") 

ind_col = list(base_df.columns)
ind_col.append("Match_list")
ind_col.append("Match_score")
ind_col.append("Actual_string")
ind_col.extend(list(raw_df.columns))

In [4]:
first_date=max(min(base_df['Arrival Date']),min(raw_df['Arrival']))
last_date=min(max(base_df['Arrival Date']),max(raw_df['Arrival']))
number_days=int((last_date-first_date)/np.timedelta64(1, 'D'))
number_days
#print(first_date,last_date,number_days)

58

In [5]:
def matching_proc(x):
    
    low_date=first_date+pd.DateOffset(x)
    high_date=low_date+pd.DateOffset(1)
    base_temp=base_df[(base_df['Arrival Date']>=low_date) & (base_df['Arrival Date']<high_date)]
    raw_temp=raw_df[(raw_df['Arrival']>=low_date) & (raw_df['Arrival']<high_date)]
    
    print(low_date,high_date,base_temp.shape, raw_temp.shape)  
    
    if((base_temp.shape[0]>0) & (raw_temp.shape[0]>0)):
    
        base_test = base_temp["Client Guestname 1"].str.strip()
        base_list = base_test.values.tolist()
        raw_test = raw_temp["Last Name"].str.strip()
        raw_list = raw_test.values.tolist()

        possibilities = []
        for string in raw_list:
            possibility = process.extractOne(string, base_list, scorer=fuzz.token_set_ratio)
            possibilities.append(possibility)
        temp_df = pd.DataFrame(possibilities, columns = ['Match_list', 'Match_score'])
        temp_df['Actual_string'] = raw_list
        temp_df['Actual_string'] = temp_df['Actual_string'].astype(str)
        temp_df.columns = ['Match_list', 'Match_score','Actual_string']

        #result_temp = pd.merge(left = temp_df[temp_df['Match_score']>=threshold], 
        result_temp = pd.merge(left = temp_df, 
            right = raw_temp, 
            left_on = ['Actual_string'], 
            right_on =['Last Name'],
            how = 'right') 
    
        result_temp2 = pd.merge(left = base_temp, 
                right = result_temp, 
                left_on = ['Client Guestname 1'], 
                right_on =['Match_list'],
                how = 'right')
 
        result_temp3 = result_temp2[result_temp2['Departure Date'] == result_temp2['Departure']].drop_duplicates()
    
        return result_temp3
    
    else:
        
        return pd.DataFrame(columns=ind_col)


In [6]:
#threshold=80
#dateshift=33
#result_tmp = matching_proc(dateshift)
#print(result_tmp.shape)
#result_tmp[['Arrival Date', 'Departure Date', 'Client Guestname 1','Arrival','Departure','Last Name','Match_score']]

In [7]:
#threshold=80

start_time = time.time()

final_result = pd.DataFrame(columns=ind_col)

for dateshift in range(number_days+1):    
#for dateshift in range(32,36):    
    print(dateshift)
    result_tmp = matching_proc(dateshift)
    print(result_tmp.shape)
    final_result = final_result.append(result_tmp, ignore_index = True) 
        
end_time = time.time()
print(end_time - start_time)
print('SUCCESS')

0
2019-12-03 00:00:00 2019-12-04 00:00:00 (1, 104) (15, 44)
(0, 151)
1
2019-12-04 00:00:00 2019-12-05 00:00:00 (0, 104) (16, 44)
(0, 151)
2
2019-12-05 00:00:00 2019-12-06 00:00:00 (3, 104) (13, 44)
(0, 151)
3
2019-12-06 00:00:00 2019-12-07 00:00:00 (0, 104) (7, 44)
(0, 151)
4
2019-12-07 00:00:00 2019-12-08 00:00:00 (3, 104) (10, 44)
(0, 151)
5
2019-12-08 00:00:00 2019-12-09 00:00:00 (1, 104) (12, 44)
(0, 151)
6
2019-12-09 00:00:00 2019-12-10 00:00:00 (1, 104) (24, 44)
(0, 151)
7
2019-12-10 00:00:00 2019-12-11 00:00:00 (5, 104) (28, 44)
(0, 151)
8
2019-12-11 00:00:00 2019-12-12 00:00:00 (3, 104) (17, 44)
(0, 151)
9
2019-12-12 00:00:00 2019-12-13 00:00:00 (0, 104) (20, 44)
(0, 151)
10
2019-12-13 00:00:00 2019-12-14 00:00:00 (1, 104) (9, 44)
(0, 151)
11
2019-12-14 00:00:00 2019-12-15 00:00:00 (6, 104) (5, 44)
(0, 151)
12
2019-12-15 00:00:00 2019-12-16 00:00:00 (11, 104) (11, 44)
(0, 151)
13
2019-12-16 00:00:00 2019-12-17 00:00:00 (6, 104) (21, 44)
(0, 151)
14
2019-12-17 00:00:00 2019-12-1

In [8]:
# del(final_result_raw)
# del(final_result_temp)
final_result_temp=final_result.copy()
final_result_temp=final_result_temp[['Match_score', 'Confirmation', 'Reservation No_', 'Last Name', 'Arrival', 'Departure']]
final_result_temp

,Match_score,Confirmation,Reservation No_,Last Name,Arrival,Departure
0,82,9671TLJ548,243228261,AAZTAARK AKIN,2019-12-20,2020-01-03
1,100,1912260537,243064833,JIANG KAI MR,2019-12-26,2020-01-10
2,100,1912260505,242578932,PETERS ALEXANDER MR,2019-12-26,2020-01-20
3,100,1912270515,242640249,NAZLIGUEL BUELENT MR,2019-12-27,2020-01-09
4,100,1912270627,238270683,MASSONE PETER MR,2019-12-27,2020-01-07
...,...,...,...,...,...,...
19303,100,2001300559,245393630,NEELIS DIRK MR,2020-01-30,2020-01-31
19304,100,2001300553,245393783,DE NOOIJER MARCEL MR,2020-01-30,2020-01-31
19305,100,2001300607,245400539,WEISL ANNABELLA MR,2020-01-30,2020-01-31
19306,100,2001300561,245499575,WARYCH DAGMARA MR,2020-01-30,2020-01-31


In [10]:
threshold = 80
final_result_temp=final_result.copy()
final_result_temp=final_result_temp[['Match_score', 'Confirmation', 'Reservation No_', 'Last Name', 'Arrival', 'Departure']]
final_result_raw = pd.merge(left = raw_org_df, right = final_result_temp, on = ['Confirmation', 'Last Name']
                            #, right_on = ['Confirmation',  'Last Name'] 
                            , how = 'left')
# final_result = final_result.drop_duplicates()
final_result_raw = final_result_raw.fillna(0)
final_result_raw['Match_status'] = final_result_raw['Match_score'].apply(lambda x: 'Y' if x > threshold else 'N')
final_result_raw['Confirmation'] = np.where(final_result_raw['Match_status'] == 'Y', final_result_raw['Reservation No_'], final_result_raw['Confirmation'])
#final_result_raw['Last Name'] = np.where(final_result_raw['Match_status'] == 'Y', final_result_raw['Client_Guestname_1'], final_result_raw['Last Name'])
final_result_raw = final_result_raw.rename(columns={'Arrival_x': 'Arrival', 'Departure_x': 'Departure', 'Arrival_y': 'Matched_Arrival', 'Departure_y': 'Matched_Departure'})
final_result_raw.to_csv(result_url,sep = ',',header=True,index=False)